In [0]:
import requests
import pandas as pd
from datetime import datetime

SERIES_CODES = {
    "XUDLUSS": "gbp_usd", "XUDLERS": "gbp_eur", "XUDLJYS": "gbp_jpy",
    "XUDLCDS": "gbp_cad", "XUDLSFS": "gbp_chf", "XUDLADS": "gbp_aud",
    "XUDLSGS": "gbp_sgd", "XUDLSKS": "gbp_sek", "XUDLDKS": "gbp_dkk",
    "IUDBEDR": "bank_rate", "IUDSOIA": "sonia",
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

def fetch_boe_data(date_from="01/Jan/2015", date_to=None):
    date_to = date_to or datetime.today().strftime("%d/%b/%Y")
    codes = ",".join(SERIES_CODES.keys())
    url = (
        "https://www.bankofengland.co.uk/boeapps/database/_iadb-fromshowcolumns.asp"
        f"?csv.x=yes&Datefrom={date_from}&Dateto={date_to}&SeriesCodes={codes}"
        "&CSVF=TN&UsingCodes=Y&VPD=Y&VFD=N"
    )
    resp = requests.get(url, headers=HEADERS, timeout=30)

    if resp.status_code == 403:
        raise RuntimeError(
            "403 Forbidden even with a browser User-Agent — likely BoE's WAF "
            "blocking Databricks' cloud IP range outright, not a header issue."
        )
    resp.raise_for_status()

    if len(resp.content) < 200:
        raise ValueError(f"Response suspiciously small ({len(resp.content)} bytes) — check series codes/date range.")

    df = pd.read_csv(pd.io.common.BytesIO(resp.content))
    df = df.rename(columns={df.columns[0]: "date"})
    return df

# --- Fetch ---
raw_df = fetch_boe_data()
print(f"Fetched {len(raw_df)} rows, columns: {list(raw_df.columns)}")

# --- Write to Delta table ---
spark_df = spark.createDataFrame(raw_df)
spark_df.write.format("delta").mode("overwrite").saveAsTable("fx_boe_data")
print("Wrote fx_boe_data (Delta table):", spark_df.count(), "rows")

# --- Write to S3 (Spark native, using Databricks' already-connected S3 access) ---
BUCKET_NAME = "fx-boe-database"  # replace with your actual bucket
s3_path = f"s3://{BUCKET_NAME}/<csvfilename>.csv"

spark_df.write.mode("overwrite").option("header", "true").csv(s3_path)
print(f"Wrote to S3")